In [1]:
!pip -q install nibabel scipy scikit-learn tqdm kagglehub wandb


In [2]:
import os
import re
import json
import random
import time
import math
from pathlib import Path

import numpy as np
import nibabel as nib
from scipy import ndimage
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print(f"Executing on device: {device}")


Executing on device: cuda


In [3]:
import wandb
wandb.login()


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: minhphanduc0 (minhphanduc0-uit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import kagglehub

DATA_ROOT = Path(
    kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
)

CACHE_DIR = Path("/kaggle/working/unet3d_optimized_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASE_FEATURES = (16, 32, 64, 128)
BOTTLENECK_FEATURES = 256
DROPOUT = 0.20

PATCH_SIZE = (128, 128, 128)
BATCH_SIZE = 2
SAMPLES_PER_VOLUME = 4
LR = 1e-4
NUM_EPOCHS = 50
VAL_EVERY = 5

NUM_WORKERS = min(4, os.cpu_count() or 2)
USE_AMP = True

CHECKPOINT_FP32 = "/kaggle/working/best_unet3d_baseline_fp32.pt"
CHECKPOINT_FP16 = "/kaggle/working/best_unet3d_baseline_fp16.pt"


Using Colab cache for faster access to the 'brats20-dataset-training-validation' dataset.


In [5]:
def find_one(case_dir, suffix):
    candidates = list(case_dir.glob("*.nii")) + list(case_dir.glob("*.nii.gz"))
    suffix = suffix.lower()
    for p in candidates:
        name = p.name.lower()
        if name.endswith(f"_{suffix}.nii") or name.endswith(f"_{suffix}.nii.gz"):
            return p
    return None

def discover_cases(root):
    cases = []
    for d in sorted(root.rglob("*")):
        if not d.is_dir():
            continue

        flair = find_one(d, "flair")
        t1 = find_one(d, "t1")
        t1ce = find_one(d, "t1ce")
        t2 = find_one(d, "t2")
        seg = find_one(d, "seg")

        if all([flair, t1, t1ce, t2, seg]):
            cases.append({
                "id": d.name,
                "flair": str(flair),
                "t1": str(t1),
                "t1ce": str(t1ce),
                "t2": str(t2),
                "seg": str(seg),
            })
    return cases

cases = discover_cases(DATA_ROOT)
print("Total: ", len(cases))

case_ids = [c["id"] for c in cases]
train_ids, val_ids = train_test_split(
    case_ids,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

train_ids = set(train_ids)
val_ids = set(val_ids)

train_cases = [c for c in cases if c["id"] in train_ids]
val_cases = [c for c in cases if c["id"] in val_ids]

print("Train: ", len(train_cases))
print("Validation: ", len(val_cases))


Total:  368
Train:  294
Validation:  74


In [6]:
def p95_normalize(vol):
    vol = vol.astype(np.float32, copy=False)
    nz = vol[vol != 0]
    if nz.size == 0:
        return np.zeros_like(vol, dtype=np.float32)

    p95 = np.percentile(nz, 95.0)
    out = np.zeros_like(vol, dtype=np.float32)
    if p95 > 0:
        mask = vol != 0
        out[mask] = vol[mask] / p95
    return out

def remap_seg(seg):
    """
    Normalize BraTS 2020 label:
    0: Background
    1: Enhancing Tumor (ET)
    2: Necrotic and Non-Enhancing Tumor Core (NCR/NET)
    3: Peritumoral Edema (ED)
    """
    out = np.zeros(seg.shape, dtype=np.uint8)
    out[seg == 4] = 1
    out[seg == 1] = 2
    out[seg == 2] = 3
    return out

def cache_case_3d(case, force=False):
    cid = case["id"]
    image_path = CACHE_DIR / f"{cid}_image_3d.npy"
    seg_path = CACHE_DIR / f"{cid}_seg_3d.npy"
    meta_path = CACHE_DIR / f"{cid}_meta_3d.json"

    if image_path.exists() and seg_path.exists() and meta_path.exists() and not force:
        return json.loads(meta_path.read_text())

    vols = []
    for key in ["flair", "t1", "t1ce", "t2"]:
        vol = nib.load(case[key]).get_fdata(dtype=np.float32)
        vols.append(p95_normalize(vol))

    seg = remap_seg(nib.load(case["seg"]).get_fdata(dtype=np.float32))

    image = np.stack(vols, axis=0).transpose(0, 3, 1, 2)
    target = seg.transpose(2, 0, 1)

    np.save(image_path, image.astype(np.float16))
    np.save(seg_path, target.astype(np.uint8))

    fg_indices = np.argwhere(target > 0)
    if len(fg_indices) > 5000:
        step = len(fg_indices) // 5000
        fg_indices = fg_indices[::step]

    counts = np.bincount(target.reshape(-1), minlength=4).astype(np.int64).tolist()

    meta = {
        "case_id": cid,
        "image_path": str(image_path),
        "seg_path": str(seg_path),
        "shape": list(target.shape),
        "fg_coords": fg_indices.tolist(),
        "class_pixel_counts": counts,
    }
    meta_path.write_text(json.dumps(meta))
    return meta

train_metas = [cache_case_3d(c) for c in tqdm(train_cases, desc="Cache 3D train")]
val_metas = [cache_case_3d(c) for c in tqdm(val_cases, desc="Cache 3D val")]


Cache 3D train:   0%|          | 0/294 [00:00<?, ?it/s]

Cache 3D val:   0%|          | 0/74 [00:00<?, ?it/s]

In [7]:
class EarlyStopping3D:
    def __init__(self, patience=3, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_metric = -float("inf") if mode == "max" else float("inf")
        self.early_stop = False

    def __call__(self, current_metric):
        if self.mode == "max":
            improved = current_metric > (self.best_metric + self.min_delta)
        else:
            improved = current_metric < (self.best_metric - self.min_delta)

        if improved:
            self.best_metric = current_metric
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False

In [10]:
class FastBraTS3DPatchDataset(Dataset):
    def __init__(self, metas, patch_size=(128, 128, 128), samples_per_volume=8, fg_prob=0.67, augment=True):
        self.metas = metas
        self.patch_size = patch_size
        self.samples_per_volume = samples_per_volume
        self.fg_prob = fg_prob
        self.augment = augment
        self.total_samples = len(metas) * samples_per_volume

    def __len__(self):
        return self.total_samples

    def _crop_patch(self, img, seg, fg_coords):
        D, H, W = seg.shape
        pD, pH, pW = self.patch_size

        if random.random() < self.fg_prob and len(fg_coords) > 0:
            center = fg_coords[random.randint(0, len(fg_coords) - 1)]
            cD, cH, cW = center
            sD = max(0, min(cD - pD // 2, D - pD))
            sH = max(0, min(cH - pH // 2, H - pH))
            sW = max(0, min(cW - pW // 2, W - pW))
        else:
            sD = random.randint(0, max(0, D - pD))
            sH = random.randint(0, max(0, H - pH))
            sW = random.randint(0, max(0, W - pW))

        patch_img = img[:, sD:sD+pD, sH:sH+pH, sW:sW+pW]
        patch_seg = seg[sD:sD+pD, sH:sH+pH, sW:sW+pW]
        return patch_img, patch_seg

    def _augment(self, x, y):
        for dim in [1, 2, 3]:
            if random.random() < 0.5:
                x = torch.flip(x, dims=[dim])
                y = torch.flip(y, dims=[dim - 1])
        if random.random() < 0.5:
            scale = random.uniform(0.9, 1.1)
            shift = random.uniform(-0.05, 0.05)
            x = torch.where(x != 0, x * scale + shift, x)
        return x, y

    def __getitem__(self, idx):
        meta_idx = idx // self.samples_per_volume
        m = self.metas[meta_idx]

        img = np.load(m["image_path"], mmap_mode="r")
        seg = np.load(m["seg_path"], mmap_mode="r")

        patch_img, patch_seg = self._crop_patch(img, seg, m.get("fg_coords", []))

        x = torch.from_numpy(np.asarray(patch_img, dtype=np.float32).copy())
        y = torch.from_numpy(np.asarray(patch_seg, dtype=np.int64).copy())

        if self.augment:
            x, y = self._augment(x, y)
        return x, y

SAMPLES_PER_VOLUME = 8

train_ds = FastBraTS3DPatchDataset(
    train_metas,
    patch_size=PATCH_SIZE,
    samples_per_volume=SAMPLES_PER_VOLUME,
    fg_prob=0.67,
    augment=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=True,
)

print(f"Total samples 3D patch: {len(train_ds)}")
print(f"Train batches / epoch: {len(train_loader)}")

Total samples 3D patch: 2352
Train batches / epoch: 1176


In [11]:
class ResBlock3D(nn.Module):
    """
    3D Residual Convolutional Block (Milletari et al., 2016; Çiçek et al., 2016)
    Sử dụng GroupNorm/InstanceNorm phù hợp cho batch size nhỏ trong 3D segmentation.
    """
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(out_ch, affine=True)
        self.relu = nn.LeakyReLU(negative_slope=0.01, inplace=True)
        self.drop = nn.Dropout3d(dropout)
        self.conv2 = nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(out_ch, affine=True)

        if in_ch != out_ch:
            self.skip = nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.relu(self.norm1(self.conv1(x)))
        out = self.drop(out)
        out = self.norm2(self.conv2(out))
        out = self.relu(out + residual)
        return out

class ResidualUNet3D(nn.Module):
    """
    Baseline 3D Residual U-Net for BraTS:
    - 4 encoder/decoder levels
    - Skip connections that directly link spatial features
    - 3D conv architecture to preserve the volumetric geometry of MRI data
    """
    def __init__(self, in_channels=4, num_classes=4, features=(16, 32, 64, 128), bottleneck=256, dropout=0.2):
        super().__init__()
        f1, f2, f3, f4 = features

        # Encoder
        self.stem = nn.Conv3d(in_channels, f1, kernel_size=3, padding=1, bias=False)
        self.enc1 = ResBlock3D(f1, f1, dropout)
        self.down1 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc2 = ResBlock3D(f1, f2, dropout)
        self.down2 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc3 = ResBlock3D(f2, f3, dropout)
        self.down3 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc4 = ResBlock3D(f3, f4, dropout)
        self.down4 = nn.MaxPool3d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = ResBlock3D(f4, bottleneck, dropout)

        # Decoder
        self.up4 = nn.ConvTranspose3d(bottleneck, f4, kernel_size=2, stride=2)
        self.dec4 = ResBlock3D(f4 + f4, f4, dropout)

        self.up3 = nn.ConvTranspose3d(f4, f3, kernel_size=2, stride=2)
        self.dec3 = ResBlock3D(f3 + f3, f3, dropout)

        self.up2 = nn.ConvTranspose3d(f3, f2, kernel_size=2, stride=2)
        self.dec2 = ResBlock3D(f2 + f2, f2, dropout)

        self.up1 = nn.ConvTranspose3d(f2, f1, kernel_size=2, stride=2)
        self.dec1 = ResBlock3D(f1 + f1, f1, dropout)

        # Output Segmentation Head
        self.head = nn.Conv3d(f1, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        e4 = self.enc4(self.down3(e3))

        b = self.bottleneck(self.down4(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head(d1)

model = ResidualUNet3D(
    in_channels=4,
    num_classes=4,
    features=BASE_FEATURES,
    bottleneck=BOTTLENECK_FEATURES,
    dropout=DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters())
mib = total_params * 4 / (1024**2)

print(f"Total params 3D Residual U-Net: {total_params:,}")
print(f"Model size (FP32): {mib:.2f} MiB")

with torch.no_grad():
    dummy = torch.zeros(1, 4, 64, 64, 64, device=device)
    out = model(dummy)
    print("Output tensor shape:", tuple(out.shape))


Total params 3D Residual U-Net: 5,741,684
Model size (FP32): 21.90 MiB
Output tensor shape: (1, 4, 64, 64, 64)


In [12]:
def compute_class_weights(metas):
    counts = np.zeros(4, dtype=np.float64)
    for m in metas:
        counts += np.asarray(m["class_pixel_counts"], dtype=np.float64)

    freq = counts / counts.sum()
    weights = 1.0 / np.sqrt(freq + 1e-12)
    weights /= weights.mean()

    print("voxel by class:", np.round(freq, 8))
    print("Class weights (WCE):", np.round(weights, 4))
    return torch.tensor(weights, dtype=torch.float32)

class_weights = compute_class_weights(train_metas).to(device)

def dice_ce_loss_3d(logits, target, class_weights=None, smooth=1e-5):
    """
    3D Multiclass Dice + Cross-Entropy Loss (Isensee et al., 2021)
    - logits: [B, C, D, H, W]
    - target: [B, D, H, W]
    """
    ce = F.cross_entropy(logits, target.long(), weight=class_weights)

    num_classes = logits.shape[1]
    probs = torch.softmax(logits, dim=1)
    target_one_hot = F.one_hot(target.long(), num_classes=num_classes).permute(0, 4, 1, 2, 3).to(dtype=probs.dtype)

    dims = (0, 2, 3, 4)
    intersection = (probs * target_one_hot).sum(dim=dims)
    denominator = probs.sum(dim=dims) + target_one_hot.sum(dim=dims)

    dice_per_class = (2.0 * intersection + smooth) / (denominator + smooth)
    dice_loss = 1.0 - dice_per_class.mean()

    return dice_loss + ce


voxel by class: [0.98886922 0.00220628 0.00245068 0.00647382]
Class weights (WCE): [0.0732 1.5505 1.4711 0.9051]


In [13]:
@torch.no_grad()
def predict_case_3d_sliding_window(model, meta, patch_size=(128, 128, 128), overlap=0.5):
    """
    Predict 3D volume with Sliding Window Patch Aggregation (Gaussian Blending) technique
    """
    model.eval()
    image = np.load(meta["image_path"], mmap_mode="r")
    C, D, H, W = image.shape
    pD, pH, pW = patch_size

    step_d = int(pD * (1 - overlap))
    step_h = int(pH * (1 - overlap))
    step_w = int(pW * (1 - overlap))

    steps_d = list(range(0, max(1, D - pD + 1), step_d))
    if steps_d[-1] + pD < D: steps_d.append(D - pD)

    steps_h = list(range(0, max(1, H - pH + 1), step_h))
    if steps_h[-1] + pH < H: steps_h.append(H - pH)

    steps_w = list(range(0, max(1, W - pW + 1), step_w))
    if steps_w[-1] + pW < W: steps_w.append(W - pW)

    prob_accum = np.zeros((4, D, H, W), dtype=np.float32)
    count_accum = np.zeros((D, H, W), dtype=np.float32)

    for sd in steps_d:
        for sh in steps_h:
            for sw in steps_w:
                sub_vol = image[:, sd:sd+pD, sh:sh+pH, sw:sw+pW]
                x = torch.from_numpy(np.asarray(sub_vol, dtype=np.float32)).unsqueeze(0).to(device)

                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                    logits = model(x)
                    probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

                prob_accum[:, sd:sd+pD, sh:sh+pH, sw:sw+pW] += probs
                count_accum[sd:sd+pD, sh:sh+pH, sw:sw+pW] += 1.0

    prob_accum /= np.maximum(count_accum, 1.0)
    pred_vol = prob_accum.argmax(axis=0).astype(np.uint8)
    return pred_vol

def dice_binary(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denom)

def brats_metrics(pred, gt):
    wt = dice_binary(pred > 0, gt > 0)
    tc = dice_binary(np.isin(pred, [1, 2]), np.isin(gt, [1, 2]))
    et = dice_binary(pred == 1, gt == 1)
    return {
        "WT": wt,
        "TC": tc,
        "ET": et,
        "Mean": (wt + tc + et) / 3.0,
    }

def evaluate_3d_dataset(model, metas):
    rows = []
    for m in tqdm(metas, desc="3D Volume Validation", leave=False):
        pred = predict_case_3d_sliding_window(model, m)
        gt = np.load(m["seg_path"], mmap_mode="r")
        rows.append(brats_metrics(pred, gt))

    keys = ["WT", "TC", "ET", "Mean"]
    return {k: float(np.mean([r[k] for r in rows])) for k in keys}


In [14]:
run = wandb.init(
    project="3du-net-baseline",
    name="brats_3d_residual_unet_final",
    config={
        "architecture": "3D Residual U-Net (Çiçek et al. / Milletari et al.)",
        "dataset": "BraTS 2020",
        "patch_size": list(PATCH_SIZE),
        "batch_size": BATCH_SIZE,
        "samples_per_volume": SAMPLES_PER_VOLUME,
        "lr": LR,
        "epochs": NUM_EPOCHS,
        "optimizer": "Adam",
        "loss": "3D Dice Cross Entropy (DiceCE)",
        "dropout": DROPOUT,
        "target_model_size_mib": 18.5,
    },
    resume="allow"
)


In [15]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
early_stopping_3d = EarlyStopping3D(patience=3, min_delta=1e-4, mode="max")
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

best_score = -1.0
best_epoch = 0

def save_checkpoint_3d(path, epoch, score):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "best_mean_dice": score,
        "architecture": "3D Residual U-Net",
        "base_features": BASE_FEATURES,
        "bottleneck": BOTTLENECK_FEATURES,
        "class_order": ["BG", "ET", "NC", "OE"],
        "loss": "3D Dice Cross Entropy",
    }
    torch.save(payload, path)

    wandb.save(path)
    print(f"Saved checkpoint to W&B: {path}")

def train_3d_baseline(epochs):
    global best_score, best_epoch

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        steps = 0
        t0 = time.time()

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device.type == "cuda")):
                logits = model(x)
                loss = dice_ce_loss_3d(logits, y, class_weights=class_weights)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_sum += float(loss.detach().item())
            steps += 1

        epoch_time = (time.time() - t0) / 60.0
        mean_train_loss = loss_sum / max(steps, 1)

        do_val = (epoch == 1 or epoch % VAL_EVERY == 0 or epoch == epochs)
        msg = f"3D U-Net | Epoch {epoch}/{epochs} | DiceCE={mean_train_loss:.5f} | Time={epoch_time:.2f} min"

        if do_val:
            metric = evaluate_3d_dataset(model, val_metas)
            msg += (
                f" | WT={metric['WT']:.4f}"
                f" TC={metric['TC']:.4f}"
                f" ET={metric['ET']:.4f}"
                f" Mean={metric['Mean']:.4f}"
            )

            wandb.log({
                "epoch": epoch,
                "train_loss": mean_train_loss,
                "val_dice_wt": metric["WT"],
                "val_dice_tc": metric["TC"],
                "val_dice_et": metric["ET"],
                "val_mean_dice": metric["Mean"],
                "epoch_time_min": epoch_time,
            })

            is_best = early_stopping_3d(metric["Mean"])
            if is_best:
                best_score = metric["Mean"]
                best_epoch = epoch
                save_checkpoint_3d(CHECKPOINT_FP32, epoch, best_score)
                wandb.save(CHECKPOINT_FP32)
                msg += " (BEST)"
            else:
                msg += f" [No improvement: {early_stopping_3d.counter}/{early_stopping_3d.patience}]"

            print(msg)

            if early_stopping_3d.early_stop:
                print(f"\n[EARLY STOP] Stopped at Epoch {epoch}")
                break
        else:
            wandb.log({
                "epoch": epoch,
                "train_loss": mean_train_loss,
                "epoch_time_min": epoch_time,
            })

        print(msg)

    return best_score, best_epoch

best_score, best_epoch = train_3d_baseline(NUM_EPOCHS)
print(f"Best Val Mean Dice: {best_score:.4f} tại Epoch {best_epoch}")


3D Volume Validation:   0%|          | 0/74 [00:00<?, ?it/s]

wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Saved checkpoint to W&B: /kaggle/working/best_unet3d_baseline_fp32.pt
3D U-Net | Epoch 1/50 | DiceCE=1.03114 | Time=11.45 min | WT=0.6958 TC=0.6321 ET=0.5857 Mean=0.6379 (BEST)
3D U-Net | Epoch 1/50 | DiceCE=1.03114 | Time=11.45 min | WT=0.6958 TC=0.6321 ET=0.5857 Mean=0.6379 (BEST)
3D U-Net | Epoch 2/50 | DiceCE=0.62997 | Time=11.35 min
3D U-Net | Epoch 3/50 | DiceCE=0.55728 | Time=11.11 min
3D U-Net | Epoch 4/50 | DiceCE=0.53563 | Time=11.10 min


3D Volume Validation:   0%|          | 0/74 [00:00<?, ?it/s]

Saved checkpoint to W&B: /kaggle/working/best_unet3d_baseline_fp32.pt
3D U-Net | Epoch 5/50 | DiceCE=0.49606 | Time=10.88 min | WT=0.8586 TC=0.8248 ET=0.7046 Mean=0.7960 (BEST)
3D U-Net | Epoch 5/50 | DiceCE=0.49606 | Time=10.88 min | WT=0.8586 TC=0.8248 ET=0.7046 Mean=0.7960 (BEST)
3D U-Net | Epoch 6/50 | DiceCE=0.47884 | Time=10.92 min
3D U-Net | Epoch 7/50 | DiceCE=0.46947 | Time=11.07 min
3D U-Net | Epoch 8/50 | DiceCE=0.44856 | Time=10.84 min
3D U-Net | Epoch 9/50 | DiceCE=0.43868 | Time=10.84 min


3D Volume Validation:   0%|          | 0/74 [00:00<?, ?it/s]

... (13 outputs left out)

3D Volume Validation:   0%|          | 0/74 [00:00<?, ?it/s]

3D U-Net | Epoch 45/50 | DiceCE=0.29052 | Time=10.92 min | WT=0.8974 TC=0.8708 ET=0.7712 Mean=0.8465 [No improvement: 1/3]
3D U-Net | Epoch 45/50 | DiceCE=0.29052 | Time=10.92 min | WT=0.8974 TC=0.8708 ET=0.7712 Mean=0.8465 [No improvement: 1/3]
3D U-Net | Epoch 46/50 | DiceCE=0.28269 | Time=10.89 min
3D U-Net | Epoch 47/50 | DiceCE=0.28338 | Time=10.88 min
3D U-Net | Epoch 48/50 | DiceCE=0.28539 | Time=11.16 min
3D U-Net | Epoch 49/50 | DiceCE=0.28380 | Time=11.04 min


3D Volume Validation:   0%|          | 0/74 [00:00<?, ?it/s]

Saved checkpoint to W&B: /kaggle/working/best_unet3d_baseline_fp32.pt
3D U-Net | Epoch 50/50 | DiceCE=0.28215 | Time=11.06 min | WT=0.9018 TC=0.8742 ET=0.7869 Mean=0.8543 (BEST)
3D U-Net | Epoch 50/50 | DiceCE=0.28215 | Time=11.06 min | WT=0.9018 TC=0.8742 ET=0.7869 Mean=0.8543 (BEST)
Best Val Mean Dice: 0.8543 tại Epoch 50


In [16]:
def postprocess_volume_3d(pred, second_lcc_min_voxels=100, et_min_voxels=50):
    pred = pred.copy()
    tumor = pred > 0
    labeled, n = ndimage.label(tumor)

    if n > 0:
        sizes = ndimage.sum(tumor, labeled, index=np.arange(1, n + 1))
        order = np.argsort(sizes)[::-1]
        keep_ids = [int(order[0] + 1)]

        if len(order) >= 2 and sizes[order[1]] >= second_lcc_min_voxels:
            keep_ids.append(int(order[1] + 1))

        keep_mask = np.isin(labeled, keep_ids)
        pred[~keep_mask] = 0

    et_count = int((pred == 1).sum())
    if 0 < et_count < et_min_voxels:
        pred[pred == 1] = 2

    return pred

ckpt = torch.load(CHECKPOINT_FP32, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print("Loaded best checkpoint from epoch:", ckpt["epoch"])

raw_val_predictions = {}
for m in tqdm(val_metas, desc="Dự đoán thô tập Val"):
    raw_val_predictions[m["case_id"]] = predict_case_3d_sliding_window(model, m)

SECOND_LCC_GRID = [0, 50, 100, 200, 500]
ET_MIN_GRID = [0, 10, 25, 50, 100, 200]

best_pp = None
best_pp_score = -1.0

for lcc_thr in SECOND_LCC_GRID:
    for et_thr in ET_MIN_GRID:
        rows = []
        for m in val_metas:
            pred = postprocess_volume_3d(
                raw_val_predictions[m["case_id"]],
                second_lcc_min_voxels=lcc_thr,
                et_min_voxels=et_thr,
            )
            gt = np.load(m["seg_path"], mmap_mode="r")
            rows.append(brats_metrics(pred, gt))

        mean = {k: float(np.mean([r[k] for r in rows])) for k in ["WT", "TC", "ET", "Mean"]}
        if mean["Mean"] > best_pp_score:
            best_pp_score = mean["Mean"]
            best_pp = {
                "second_lcc_min_voxels": lcc_thr,
                "et_min_voxels": et_thr,
                **mean,
            }

print("Post-processing result:", best_pp)
wandb.log({
    "postprocess_best_mean_dice": best_pp["Mean"],
    "postprocess_best_wt": best_pp["WT"],
    "postprocess_best_tc": best_pp["TC"],
    "postprocess_best_et": best_pp["ET"],
})


Loaded best checkpoint from epoch: 50


Dự đoán thô tập Val:   0%|          | 0/74 [00:00<?, ?it/s]

Post-processing result: {'second_lcc_min_voxels': 500, 'et_min_voxels': 50, 'WT': 0.9025704510449865, 'TC': 0.8742188267930782, 'ET': 0.8004246162910961, 'Mean': 0.8590712980430536}


In [17]:
final_fp32 = "/kaggle/working/unet3d_final_fp32.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "base_features": BASE_FEATURES,
    "bottleneck": BOTTLENECK_FEATURES,
    "best_mean_dice": best_score,
    "postprocess": best_pp,
    "class_order": ["BG", "ET", "NC", "OE"],
}, final_fp32)
wandb.save(final_fp32)

fp16_state = {k: v.detach().cpu().half() for k, v in model.state_dict().items()}
final_fp16 = CHECKPOINT_FP16
torch.save({
    "model_state_dict": fp16_state,
    "base_features": BASE_FEATURES,
    "bottleneck": BOTTLENECK_FEATURES,
    "best_mean_dice": best_score,
    "postprocess": best_pp,
    "dtype": "float16",
    "class_order": ["BG", "ET", "NC", "OE"],
}, final_fp16)
wandb.save(final_fp16)

def file_mib(path):
    return Path(path).stat().st_size / 1024**2

print("=" * 60)
print("COMPARISON BASELINE 3D RESIDUAL U-NET:")
print("=" * 60)
print(f"Total weights: {sum(p.numel() for p in model.parameters()):,}")
print(f"Checkpoint FP32 (.pth): {file_mib(final_fp32):.2f} MiB")
print(f"Checkpoint FP16 (.pt): {file_mib(final_fp16):.2f} MiB")
print(f"Best Mean Dice (before post-processing): {best_score:.4f}")
print(f"Best Mean Dice (after post-processing): {best_pp['Mean']:.4f}")
print("=" * 60)
wandb.finish()


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


COMPARISON BASELINE 3D RESIDUAL U-NET:
Total weights: 5,741,684
Checkpoint FP32 (.pth): 21.93 MiB
Checkpoint FP16 (.pt): 10.97 MiB
Best Mean Dice (before post-processing): 0.8543
Best Mean Dice (after post-processing): 0.8591


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
epoch_time_min,█▇▄▄▂▁▁▂▃▂▂▂▃▃▃▂▃▃▂▂▂▃▃▄▃▃▅▃▂▂▃▂▃▄▃▂▂▁▅▄
postprocess_best_et,▁
postprocess_best_mean_dice,▁
postprocess_best_tc,▁
postprocess_best_wt,▁
train_loss,█▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dice_et,▁▅▅▆▇█▇▇█▇█
val_dice_tc,▁▇▇▇███████
val_dice_wt,▁▇▆▇█▇█████
+1,...
